# Diamonds case study (original group notebook)

Course project for Data Analytics at KIT/FZI, kept here as the starting point of this repo.
Course hand-in instructions have been removed; the analysis is unchanged.
The rebuilt, tested version lives in `src/diamonds/`.


# STUDENT DETAILS
Nurul Fazlina Zainal Abidin, Shituan Lin, Medha Agarwal, Hongyi Zhao
- 

## The diamonds dataset

### The raw data

Here is a short description of what is in the dataset:
The dataset contains the prices and other attributes of almost 54,000 diamonds. 

- `price`: price in US dollars
- `carat`: weight of the diamond
- `cut`: quality of the cut (Fair, Good, Very Good, Premium, Ideal)
- `color`: diamond colour, from J (worst) to D (best)
- `clarity`: a measurement of how clear the diamond is (I1 (worst), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (best))
- `x`: length in mm
- `y`: width in mm
- `z`: depth in mm
- `depth`: total depth percentage = z / mean(x, y) = 2 * z / (x + y)
- `table`: width of top of diamond relative to widest point


## YOUR CODE

## I. Download and import the data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn import set_config
set_config(display="text")
from sklearn import tree
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor


In [ ]:
# LOAD AND UNDERSTAND THE DATA

diamonds = sns.load_dataset("diamonds")

print(diamonds.head())       # display first rows
print(diamonds.info())       # display information

In [ ]:
# check data types and missing values
print(diamonds.isnull().sum())

In [ ]:
# Descriptive statistics for numerical variables
print(diamonds.describe(include="all"))

In [ ]:
# Frequency of categorical variables
for column in ["cut", "color", "clarity"]:
    print("\nFrequency of", column)
    print(diamonds[column].value_counts())

In [ ]:
# Automatically report the main characteristics
print("KEY FINDINGS\n")

print("Price range:", diamonds["price"].min(),
      "to", diamonds["price"].max(), "USD")
print("Average price:", round(diamonds["price"].mean(), 2), "USD")
print("Median price:", round(diamonds["price"].median(), 2), "USD")
print("Carat range:", diamonds["carat"].min(),
      "to", diamonds["carat"].max())
print("Missing values in the complete dataset:",
      diamonds.isnull().sum().sum())

In [ ]:
diamonds['cut'].unique()

In [ ]:
diamonds['color'].unique()

In [ ]:
diamonds['clarity'].unique()

**First impression**: the dataset has about 54,000 rows (one diamond per row) and 10 columns.
- 7 numerical columns: `carat`, `depth`, `table`, `price`, `x`, `y`, `z`
- 3 categorical columns: `cut`, `color`, `clarity` -- these are *ordered* categories (e.g. Fair < Good < Very Good < Premium < Ideal), so their order carries information.

# II. Data Prepocessing and Anomaly Checks

In [ ]:
# Check exact duplicate rows before making any changes
duplicate_count = diamonds.duplicated().sum()

print("\nFINDINGS - DATA QUALITY")
print("Exact duplicate rows:", duplicate_count)

# Count observations with zero or negative physical dimensions
invalid_dimensions = (
    (diamonds["x"] <= 0) |
    (diamonds["y"] <= 0) |
    (diamonds["z"] <= 0)
)

print("Rows with zero or negative dimensions:",
      invalid_dimensions.sum())

# Check extreme measurements that deserve investigation
print("Maximum x dimension:", diamonds["x"].max())
print("Maximum y dimension:", diamonds["y"].max())
print("Maximum z dimension:", diamonds["z"].max())
print("Maximum depth percentage:", diamonds["depth"].max())
print("Maximum table percentage:", diamonds["table"].max())


Findings:
- There are **no missing values** (`NaN`), so `dropna` is not necessary here (in contrast to the penguins).
- There are **duplicated rows**.
- Size (`x`, `y` and `z`) has some **negative values or 0** -- a diamond cannot have negative or zero dimension. These are missing/invalid measurements that were encoded as negative or 0.
- The maximum of `y` (~59 mm) and `z` (~32 mm) is far away from the rest of the data (75% of the diamonds have `y` below ~7 mm) -- most likely typing errors.

In [ ]:
# Remove invalid physical dimensions.
# This is a modelling assumption: identical records may also represent different diamonds with identical measured attributes.
diamonds_clean = diamonds.copy()
diamonds_clean = diamonds_clean.drop_duplicates()
diamonds_clean = diamonds_clean[
    (diamonds_clean["x"] > 0) &
    (diamonds_clean["y"] > 0) &
    (diamonds_clean["z"] > 0)
]

print("\nRows before cleaning:", len(diamonds))
print("Rows after cleaning:", len(diamonds_clean))
print("Rows removed:", len(diamonds) - len(diamonds_clean))

# Check data quality again
print("\nMissing values after cleaning:")
print(diamonds_clean.isnull().sum())

Most rows are consistent (the formula holds), and the few rows that deviate strongly indicate measurement errors in `x`, `y` or `z`. 

**Cleaning decisions**
1. Remove duplicate rows.
2. Remove rows where `x`, `y` or `z` is negative.

All of this is a tiny share of the data, so we lose almost no information.

# III. Visual impression of the data

In [ ]:
sns.pairplot(data=diamonds_clean)

In [ ]:
# Skipped Indirect Diagram
# sns.pairplot(data=diamonds, hue='color')
# sns.pairplot(data=diamonds, hue='cut')
# sns.pairplot(data=diamonds, hue='clarity')

# IV. Relationship between values

In [ ]:
from sklearn.linear_model import LinearRegression

## i. Relationship between carat and price

In [ ]:
diamonds_clean[['carat', 'price']]
model_carat_price = LinearRegression()
X = diamonds_clean[['carat']]
y = diamonds_clean['price']
model_carat_price.fit(X,y)
model_carat_price.score(X, y)

In [ ]:
sns.scatterplot(data=diamonds_clean, x="carat", y="price")
sns.lineplot(x=X['carat'], y=model_carat_price.predict(X), color="red")

Conclusion: price and carat has postive relationship, which means weight of the diamond affects diamond price.

## ii. Relationship between carat and x

In [ ]:
diamonds_clean[['carat', 'x']]

model_carat_x = LinearRegression()
X = diamonds_clean[['carat']]
y = diamonds_clean['x']

model_carat_x.fit(X, y)
model_carat_x.score(X, y)

In [ ]:
sns.scatterplot(data=diamonds_clean, x="carat", y="x")
sns.lineplot(x=X['carat'], y=model_carat_x.predict(X), color="red")

Conclusion: Carat and x show a positive relationship. Diamonds with greater weight generally have a greater length (x). This indicates that carat and x contain related information about diamond size.

## iii. Relationship between Carat and y

In [ ]:
diamonds_clean[['carat', 'y']]

model = LinearRegression()
X = diamonds_clean[['carat']]
y = diamonds_clean['y']

model.fit(X, y)
model.score(X, y)

In [ ]:
sns.scatterplot(data=diamonds_clean, x="carat", y="y")
sns.lineplot(x=X['carat'], y=model.predict(X), color='red', errorbar=None)


Conclusion: y and carat has postive relationship, which means weight of the diamond affects diamond width.

## iv. Relationship between carat and z

In [ ]:
diamonds_clean[['carat', 'z']]

model_carat_z = LinearRegression()
X = diamonds_clean[['carat']]
y = diamonds_clean['z']

model_carat_z.fit(X, y)
model_carat_z.score(X, y)

In [ ]:
sns.scatterplot(data=diamonds_clean, x="carat", y="z")
sns.lineplot(x=X['carat'], y=model_carat_z.predict(X), color="red")

Conclusion: Carat and z generally show a positive relationship. Heavier diamonds tend to have greater physical depth (z). This suggests that carat and z also contain related information about diamond size.

## v. Relationship between x and price

In [ ]:
diamonds_clean[['x', 'price']]

model_x_price = LinearRegression()
X = diamonds_clean[['x']]
y = diamonds_clean['price']

model_x_price.fit(X, y)
model_x_price.score(X, y)

In [ ]:
sns.scatterplot(data=diamonds_clean, x="x", y="price")
sns.lineplot(x=X['x'], y=model_x_price.predict(X), color="red")

Conclusion: Diamond length (x) and price generally show a positive relationship. Diamonds with greater length tend to be more expensive. However, the relationship is not perfectly linear, suggesting that other factors also influence price.

## vi. Relationship between y and price

In [ ]:
diamonds_clean[['y', 'price']]

model_y_price = LinearRegression()
X = diamonds_clean[['y']]
y = diamonds_clean['price']

model_y_price.fit(X, y)
model_y_price.score(X, y)

In [ ]:
sns.scatterplot(data=diamonds_clean, x="y", y="price")
sns.lineplot(x=X['y'], y=model_y_price.predict(X), color="red")

Conclusion: Diamond width (y) generally has a positive association with price. However, unusual measurements and outliers may weaken the linear relationship. Therefore, y alone may not be a reliable predictor of diamond price.

## vii. Relationship between z and price

In [ ]:
diamonds_clean[['z', 'price']]

model_z_price = LinearRegression()
X = diamonds_clean[['z']]
y = diamonds_clean['price']

model_z_price.fit(X, y)
model_z_price.score(X, y)

In [ ]:
sns.scatterplot(data=diamonds_clean, x="z", y="price")
sns.lineplot(x=X['z'], y=model_z_price.predict(X), color="red")

Conclusion: Diamond depth (z) generally has a positive relationship with price. However, the presence of outliers and the variation in prices suggest that z alone does not fully explain diamond price.

## viii. Relationship between x and y

In [ ]:
diamonds_clean[['x', 'y']]

model_x_y = LinearRegression()
X = diamonds_clean[['x']]
y = diamonds_clean['y']

model_x_y.fit(X, y)
model_x_y.score(X, y)

In [ ]:
sns.scatterplot(data=diamonds_clean, x="x", y="y")
sns.lineplot(x=X['x'], y=model_x_y.predict(X), color="red")

Conclusion: Diamond length (x) and width (y) generally have a strong positive relationship. As diamond length increases, width also tends to increase. This indicates that x and y contain similar information about diamond size and may be redundant when predicting price.

## ix. Relationship between x and z

In [ ]:
diamonds_clean[['x', 'z']]

model_x_z = LinearRegression()
X = diamonds_clean[['x']]
y = diamonds_clean['z']

model_x_z.fit(X, y)
model_x_z.score(X, y)

In [ ]:
sns.scatterplot(data=diamonds_clean, x="x", y="z")
sns.lineplot(x=X['x'], y=model_x_z.predict(X), color="red")

Conclusion: Diamond length (x) and physical depth (z) generally have a positive relationship. Longer diamonds tend to be deeper. This suggests that x and z may contain overlapping information about diamond dimensions.

## x. Relationship between x, y, z and price 

In [ ]:
# new: separate dataset for training and testing
from sklearn.model_selection import train_test_split
X = diamonds_clean[['x', 'y', 'z']]
y = diamonds_clean['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

model_xyz_price = LinearRegression()
model_xyz_price.fit(X_train, y_train)

In [ ]:
print("Training R²:", model_xyz_price.score(X_train, y_train))

In [ ]:
print("Testing R²:", model_xyz_price.score(X_test, y_test))

Conclusion: The analysis suggests that carat, x, y, and z are positively related because they represent different aspects of diamond size. The relationships between physical dimensions also indicate that some features contain overlapping information.
Carat and the physical dimensions are useful candidates for predicting diamond price. However, price is not determined solely by size, and the relationships are not perfectly linear. Therefore, additional features such as cut, color, and clarity should be considered when developing a prediction model.

# V. Main drivers of diamond prices model

## i.Clustering

In [ ]:
X_cluster = diamonds_clean[["x", "y", "z", "carat"]]

kmeansmodel = KMeans(n_clusters=3, random_state=42)
kmeansmodel.fit(X_cluster)
raw_labels = kmeansmodel.predict(X_cluster)

sns.scatterplot(x=diamonds_clean["x"], y=diamonds_clean["price"], hue=raw_labels)

In [ ]:
sns.scatterplot(x=diamonds_clean["y"], y=diamonds_clean["price"], hue=raw_labels)

In [ ]:
sns.scatterplot(x=diamonds_clean["z"], y=diamonds_clean["price"], hue=raw_labels)

In [ ]:
sns.scatterplot(x=diamonds_clean["carat"], y=diamonds_clean["price"], hue=raw_labels)

In [ ]:
scaler = StandardScaler()
pipeline = make_pipeline(scaler, KMeans(n_clusters=3, random_state=42))
pipeline.fit(X_cluster)
scaled_labels = pipeline.predict(X_cluster)

sns.scatterplot(x=diamonds_clean["x"], y=diamonds_clean["price"], hue=scaled_labels)

In [ ]:
sns.scatterplot(x=diamonds_clean["y"], y=diamonds_clean["price"], hue=scaled_labels)

In [ ]:
sns.scatterplot(x=diamonds_clean["z"], y=diamonds_clean["price"], hue=scaled_labels)

In [ ]:
sns.scatterplot(x=diamonds_clean["carat"], y=diamonds_clean["price"], hue=scaled_labels)

In [ ]:
check_df = pd.DataFrame({"unscaled_cluster": raw_labels,
                         "scaled_cluster": scaled_labels})
check_df

## ii. Elbow method on the full cleaned inputs

In [ ]:
n_clusters = range(1, 10)
scores = []
for n in n_clusters:
    model = make_pipeline(StandardScaler(), KMeans(n_clusters=n, random_state=42))
    model.fit_predict(X_cluster)
    scores.append(model[1].inertia_)

sns.lineplot(x=n_clusters, y=scores)

## iii.Price versus x

In [ ]:
X = diamonds_clean[["x", "y", "z", "carat"]]
y = diamonds_clean["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

tree_models = {}
forest_models = {}
tree_predictions = {}
forest_predictions = {}
results = []

In [ ]:
X_train_x = X_train[["x"]]
X_test_x = X_test[["x"]]

tree_x = tree.DecisionTreeRegressor(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)
tree_x.fit(X_train_x, y_train)

forest_x = RandomForestRegressor(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)
forest_x.fit(X_train_x, y_train)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(16, 8), dpi=100)
tree.plot_tree(tree_x, feature_names=list(X_train_x.columns), filled=True)
plt.show()

## iv. price versus y

In [ ]:
X_train_y = X_train[["y"]]
X_test_y = X_test[["y"]]

tree_y = tree.DecisionTreeRegressor(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)
tree_y.fit(X_train_y, y_train)

forest_y = RandomForestRegressor(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)
forest_y.fit(X_train_y, y_train)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(16, 8), dpi=100)
tree.plot_tree(tree_y, feature_names=list(X_train_y.columns), filled=True)
plt.show()


## v.price versus z

In [ ]:
X_train_z = X_train[["z"]]
X_test_z = X_test[["z"]]

tree_z = tree.DecisionTreeRegressor(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)
tree_z.fit(X_train_z, y_train)

forest_z = RandomForestRegressor(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)
forest_z.fit(X_train_z, y_train)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(16, 8), dpi=100)
tree.plot_tree(tree_z, feature_names=list(X_train_z.columns), filled=True)
plt.show()

## vi. price versus carat

In [ ]:
X_train_carat = X_train[["carat"]]
X_test_carat = X_test[["carat"]]

tree_carat = tree.DecisionTreeRegressor(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)
tree_carat.fit(X_train_carat, y_train)

forest_carat = RandomForestRegressor(
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)
forest_carat.fit(X_train_carat, y_train)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(16, 8), dpi=100)
tree.plot_tree(tree_carat, feature_names=list(X_train_carat.columns), filled=True)
plt.show()

## vii. Predict a new diamond, separately

In [ ]:
forest_models = {
    "x": forest_x,
    "y": forest_y,
    "z": forest_z,
    "carat": forest_carat
}
new_diamonds_clean = pd.DataFrame({"x": [6.4], "y": [6.4], "z": [4.0], "carat": [1.0]})
new_predictions = []
for feature in ["x", "y", "z", "carat"]:
    prediction = forest_models[feature].predict(new_diamonds_clean[[feature]])
    new_predictions.append({"feature": feature, "predicted_price_USD": prediction[0]})
print(pd.DataFrame(new_predictions))